In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.gridspec import GridSpec
import seaborn as sns
import pandas as pd
import random
import hydra
import lightning as lit
import torch
import torch.nn as nn
import wandb
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import MinMaxScaler
from lightning.pytorch.callbacks import LearningRateMonitor, EarlyStopping, RichProgressBar, ModelCheckpoint
from lightning.pytorch.loggers import WandbLogger, CSVLogger
from loguru import logger
from omegaconf import DictConfig, OmegaConf
from torch.utils.data import DataLoader
from hydra import initialize, initialize_config_module, initialize_config_dir, compose
from omegaconf import OmegaConf

from datetime import datetime


from machine_intelligence.utilities.viz_setup import figure_setup_grid

from machine_intelligence.utilities.viz_setup import figure_setup, five_min_to_hours
import matplotlib.ticker as ticker

from machine_intelligence.data.SepsisPredDataModule import SepsisPredDataModule
# from machine_intelligence.interpretability.shap_analysis import ModelShapWrapper


from machine_intelligence.utilities.viz_setup import figure_setup, VariableNames


In [ ]:

cwd = Path.cwd()
cwd

In [ ]:

project_root = Path(cwd).parents[1]
project_root

In [ ]:

current_date = datetime.now().strftime("%Y-%m-%d")
plot_output_dir = project_root / "plot_outputs" / current_date
plot_output_dir.mkdir(parents=True, exist_ok=True)

In [ ]:

with initialize(version_base=None, config_path="../../configs"):
    cfg = compose(config_name="run_training.yaml", overrides=["data=cohort_a_r0.4_imp_none"])

    print(cfg)


In [ ]:
wandb.init(mode="disabled")


In [ ]:
sepsis_pred_data = SepsisPredDataModule(cfg=cfg)


In [ ]:
sepsis_pred_data.setup("test")

In [ ]:
filtering_dl = sepsis_pred_data.test_dataloader()
batch = next(iter(filtering_dl))

In [ ]:
batch_ts, batch_static, batch_label = batch

sample_ts_1 = batch_ts[6]

In [ ]:
df_train = sepsis_pred_data.df_train

In [ ]:
df_sample_ts_1 = df_train.loc[df_train[sepsis_pred_data.sample_index_column] == 6]
df_sample_ts_1 = df_sample_ts_1.drop(
    columns=[sepsis_pred_data.sample_index_column, "time_offset", "patientunitstayid", "sepsis_patient",
             "sepsis_present", "sepsis_sample", "sepsis_sample_shifted"])
df_sample_ts_1

In [ ]:
columns_for_vis = ["heartrate", "respiration", "sao2", "labname_BUN", "labname_bedside glucose"]

In [ ]:
zero_columns = df_sample_ts_1.loc[:, (df_sample_ts_1 == 0).all()].columns.to_list()
df_sample_ts_1 = df_sample_ts_1.drop(columns=zero_columns).reset_index(drop=True)
non_zero_columns = df_sample_ts_1.columns.to_list()

In [ ]:
df_sample_ts_1 = df_sample_ts_1.ffill()

df_sample_ts_1 = df_sample_ts_1.replace(to_replace=np.nan, value=float(0))


In [ ]:
from scipy import signal

# from signal import sosfiltfilt
# sampling frequency of 5 mins in Hz  -> 1 / 60 Hz
fs = 1 / (60 * 5)

nyquist = 0.5 * fs  # f_nyquist = 0.5 * f_sampling
cutoff = 0.0005  #(1 / 60) * 55  # 1 / 60 Hz * x
normal_cutoff = cutoff / nyquist
print(f"{normal_cutoff=}")

sos = signal.butter(1, normal_cutoff, btype="lowpass", output="sos")

np_df_filtered = signal.sosfiltfilt(sos, df_sample_ts_1, axis=0)
np_df_filtered

In [ ]:

df_sample_ts_1_filtered = pd.DataFrame(data=np_df_filtered, columns=non_zero_columns)
df_sample_ts_1_filtered


In [ ]:
df_sample_ts_1.head()

In [ ]:
fig, ax, font_labels, font_annots = figure_setup(num_cols=1, num_row=1, height_cm=8, width_cm=8)

# sns.lineplot(df_sample_ts_1.loc[:, ["heartrate", "systemicdiastolic", "respiration", "noninvasivesystolic", "labname_MCH", "labname_-lymphs"]], ax=ax, palette=sns.color_palette("cubehelix"), dashes=False)
sns.lineplot(df_sample_ts_1.loc[:, columns_for_vis], ax=ax, palette=sns.color_palette("plasma", n_colors=5), dashes=False, alpha=0.5)
# sns.lineplot(df_sample_ts_1_filtered.loc[:, ["heartrate", "systemicdiastolic", "respiration", "noninvasivesystolic", "labname_MCH", "labname_-lymphs"]], ax=ax,
sns.lineplot(df_sample_ts_1_filtered.loc[:, columns_for_vis], ax=ax,
             palette=sns.color_palette("plasma", n_colors=5), linestyle="--", dashes=False)

five_min_hour_formatter = ticker.FuncFormatter(func=five_min_to_hours)
ax.xaxis.set_major_locator(plt.MultipleLocator(1 * 12))  # 5 min  x 12
ax.xaxis.set_major_formatter(five_min_hour_formatter)

handles, labels = ax.get_legend_handles_labels()
human_readable_labels = [VariableNames.get_pretty_name(label) for label in labels]

ax.legend(handles, human_readable_labels,
          loc="upper center",
          bbox_to_anchor=(0.5, -0.3),
          ncol=2,
          frameon=False,
          fancybox=True,
          shadow=True,
          prop=font_annots,
          )


ax.set_xlabel("Time [h]")

ax.set_ylabel("Time Series Variables",
              )
ax.set_title("Butterworth") 

ax.margins(x=0)


plt.tight_layout()

if save_as_svg := True:
    fig.savefig(plot_output_dir / "low pass filtered ts.svg", format="svg")



### Moving Average

In [ ]:
df_sample_ts_1_ma = df_sample_ts_1.rolling(window=5, min_periods=1).mean()

In [ ]:
fig, ax, font_labels, font_annots = figure_setup(num_cols=1, num_row=1, height_cm=8, width_cm=8)

# sns.lineplot(df_sample_ts_1.loc[:, ["heartrate", "systemicdiastolic", "respiration", "noninvasivesystolic", "labname_MCH", "labname_-lymphs"]], ax=ax, palette=sns.color_palette("cubehelix"), dashes=False)
sns.lineplot(df_sample_ts_1.loc[:, columns_for_vis], ax=ax, palette=sns.color_palette("plasma", n_colors=5), dashes=False, alpha=0.5)
# sns.lineplot(df_sample_ts_1_filtered.loc[:, ["heartrate", "systemicdiastolic", "respiration", "noninvasivesystolic", "labname_MCH", "labname_-lymphs"]], ax=ax,
sns.lineplot(df_sample_ts_1_ma.loc[:, columns_for_vis], ax=ax,
             palette=sns.color_palette("plasma", n_colors=5), linestyle="--", dashes=False, )

five_min_hour_formatter = ticker.FuncFormatter(func=five_min_to_hours)
ax.xaxis.set_major_locator(plt.MultipleLocator(1 * 12))  # 5 min  x 12
ax.xaxis.set_major_formatter(five_min_hour_formatter)

handles, labels = ax.get_legend_handles_labels()
human_readable_labels = [VariableNames.get_pretty_name(label) for label in labels]

ax.legend(handles, human_readable_labels,
          loc="upper center",
          bbox_to_anchor=(0.5, -0.3),
          ncol=2,
          frameon=False,
          fancybox=True,
          shadow=True,
          prop=font_annots,
          )


ax.set_xlabel("Time [h]")

ax.set_ylabel("Time Series Variables",
              )
ax.set_title("Moving Average") 

ax.margins(x=0)

plt.tight_layout()

if save_as_svg := True:
    fig.savefig(plot_output_dir / "regular moving averaged ts.svg", format="svg")



### Exponential Moving Average

In [ ]:
df_sample_ts_1_ma = df_sample_ts_1.rolling(window=5, min_periods=1, win_type="exponential").mean()
fig, ax, font_labels, font_annots = figure_setup(num_cols=1, num_row=1, height_cm=8, width_cm=8)

# sns.lineplot(df_sample_ts_1.loc[:, ["heartrate", "systemicdiastolic", "respiration", "noninvasivesystolic", "labname_MCH", "labname_-lymphs"]], ax=ax, palette=sns.color_palette("cubehelix"), dashes=False)
sns.lineplot(df_sample_ts_1.loc[:, columns_for_vis], ax=ax, palette=sns.color_palette("plasma", n_colors=5), dashes=False, alpha=0.5)
# sns.lineplot(df_sample_ts_1_filtered.loc[:, ["heartrate", "systemicdiastolic", "respiration", "noninvasivesystolic", "labname_MCH", "labname_-lymphs"]], ax=ax,
sns.lineplot(df_sample_ts_1_ma.loc[:, columns_for_vis], ax=ax,
             palette=sns.color_palette("plasma", n_colors=5), linestyle="--", dashes=False, )

five_min_hour_formatter = ticker.FuncFormatter(func=five_min_to_hours)
ax.xaxis.set_major_locator(plt.MultipleLocator(1 * 12))  # 5 min  x 12
ax.xaxis.set_major_formatter(five_min_hour_formatter)

handles, labels = ax.get_legend_handles_labels()
human_readable_labels = [VariableNames.get_pretty_name(label) for label in labels]

ax.legend(handles, human_readable_labels,
          loc="upper center",
          bbox_to_anchor=(0.5, -0.3),
          ncol=2,
          frameon=False,
          fancybox=True,
          shadow=True,
          prop=font_annots,
          )


ax.set_xlabel("Time [h]")

ax.set_ylabel("Time Series Variables",
              )
ax.set_title("Exponential Moving Average") 


ax.margins(x=0)

plt.tight_layout()

if save_as_svg := True:
    fig.savefig(plot_output_dir / "exponential moving averaged ts.svg", format="svg")


In [ ]:
import pywt


def pad_signal_with_same(data, pad_length):
    start_pad = np.full(pad_length, data[0])
    end_pad = np.full(pad_length, data[-1])
    padded_data = np.concatenate([start_pad, data, end_pad])
    return padded_data


def wavelet_denoising(data, wavelet="db5", level=1, pad_length=10):
    padded_data = pad_signal_with_same(data, pad_length)
    # coeffs = pywt.wavedec(data, wavelet, mode="per")
    coeffs = pywt.wavedec(padded_data, wavelet, mode="per")
    # sigma = (1/0.6745) * np.median(np.abs(coeffs[-level] - np.median(coeffs[-level])))
    sigma = (1 / 0.6745) * np.mean(np.abs(coeffs[-level] - np.mean(coeffs[-level])))
    
    if np.all(coeffs[-level] == 0):
        
        return data
        
    else:
        sigma = (1 / 0.6745) * np.mean(np.abs(coeffs[-level] - np.mean(coeffs[-level])))
        uthresh = sigma * np.sqrt(2 * np.log(len(padded_data)))
        
    # uthresh = sigma * np.sqrt(2 * np.log(len(data)))
    coeffs[1:] = (pywt.threshold(i, value=uthresh, mode="soft") for i in coeffs[1:])
    denoised_padded_data = pywt.waverec(coeffs, wavelet, mode="per")
    denoised_data = denoised_padded_data[pad_length:-pad_length]
    denoised_data[denoised_data < 0] = 0 # related to step changes on zero filled values (< 0 is unphysiological)
    return denoised_data



pywt.wavelist(kind="discrete")

In [ ]:
np_sample_heartrate_cwt = wavelet_denoising(df_sample_ts_1["heartrate"].to_numpy(), wavelet="bior6.8", level=1)


In [ ]:
df_sample_ts_1_cwt = pd.DataFrame(data=np_sample_heartrate_cwt, columns=["heartrate"])

df_sample_ts_1_cwt

In [ ]:
def apply_wavelet_denoising_to_dataframe(df, wavelet="db5", level=1, pad_length=10):
    denoised_df = pd.DataFrame(index=df.index)
    for column in df.columns:
        denoised_data = wavelet_denoising(df[column].to_numpy(), wavelet, level, pad_length=pad_length)
        denoised_data = denoised_data[:len(df)]
        denoised_df[column] = denoised_data
    return denoised_df


df_sample_ts_1_cwt_all = apply_wavelet_denoising_to_dataframe(df=df_sample_ts_1, wavelet="db8", level=1, pad_length=10)
df_sample_ts_1_cwt_all.head()

In [ ]:

df_sample_cwt_all_prt = VariableNames.get_pretty_dataframe(df_sample_ts_1_cwt_all)
df_sample_cwt_all_prt.head()

In [ ]:
fig, ax, font_labels, font_annots = figure_setup(num_cols=1, num_row=1, height_cm=8, width_cm=8)



# sns.lineplot(df_sample_ts_1.loc[:, ["heartrate", "systemicdiastolic", "respiration", "noninvasivesystolic", "labname_MCH", "labname_-lymphs"]], ax=ax, palette=sns.color_palette("cubehelix"), dashes=False)
sns.lineplot(df_sample_ts_1.loc[:, columns_for_vis], ax=ax, palette=sns.color_palette("plasma", n_colors=5), dashes=False, alpha=0.5)
# sns.lineplot(df_sample_ts_1_filtered.loc[:, ["heartrate", "systemicdiastolic", "respiration", "noninvasivesystolic", "labname_MCH", "labname_-lymphs"]], ax=ax,
sns.lineplot(df_sample_ts_1_cwt_all.loc[:, columns_for_vis], ax=ax,
             palette=sns.color_palette("plasma", n_colors=5), linestyle="--", dashes=False)



five_min_hour_formatter = ticker.FuncFormatter(func=five_min_to_hours)
ax.xaxis.set_major_locator(plt.MultipleLocator(1 * 12))  # 5 min  x 12
ax.xaxis.set_major_formatter(five_min_hour_formatter)

handles, labels = ax.get_legend_handles_labels()
human_readable_labels = [VariableNames.get_pretty_name(label) for label in labels]

ax.legend(handles, human_readable_labels,
           loc="upper center",
    bbox_to_anchor=(0.5, -0.3),
    ncol=2,
    frameon=False,
    fancybox=True,
    shadow=True,
    prop=font_annots,
          )


ax.set_xlabel("Time [h]")

ax.set_ylabel("Time Series Variables",
              )
ax.set_title("Wavelet") 


ax.margins(x=0)

plt.tight_layout()

if save_as_svg := True:
    fig.savefig(plot_output_dir / "wavelet_filtered_ts.svg", format="svg")


There's some ringing going on but thats expected when a step function shaped change takes place, the temporal shift is however much better than with moving averages!
But comparable to a butterworth filter, needs some experimentation to take a look what works better